# 1. 利用KNN算法实现车型推荐



## 0. 载入相关模块

In [ ]:
import pandas as pd
import sklearn.neighbors as neighbors

## 1. 数据预处理

### 1.1 载入数据

In [14]:
df = pd.read_csv("原始-基础数据.csv", encoding="gbk")
df

,型号,长mm,宽mm,高mm
0,BAOJ310-,4032,1680,1450
1,XW-M1+,4330,1535,1885
2,GoF,4259,1799,1452
3,CrV,4585,1855,1679
4,BZ-08,4590,1820,1488
5,jeta,4501,1704,1469
6,POL-,4053,1740,1449
7,BW5+,5087,1868,1500
8,BDe6+,4560,1822,1645
9,RWi5,4601,1818,1489


### 1.2 数据预处理

1. 区分已知数据和待预测数据
2. 划分训练集和测试集

In [31]:
# (1) 创建新列“推荐”，用于存储推荐结果
df["推荐"] = None

# (2) 获取所有型号的推荐信息
# 已知数据：型号后缀为+或-；+ 代表推荐，- 代表不推荐
# 未知数据：型号后缀为空，代表未知

for name in df["型号"].values:
    if name[-1] == "+":
        df.loc[df["型号"] == name, "推荐"] = 1
    elif name[-1] == "-":
        df.loc[df["型号"] == name, "推荐"] = 0

unknown_df = df[df["推荐"].isnull()]
known_df = df[~df["推荐"].isnull()]
# 指定known_df推荐列为整数类型
known_df["推荐"] = known_df["推荐"].astype(int)

display(unknown_df)
display(known_df)

,型号,长mm,宽mm,高mm,推荐
2,GoF,4259,1799,1452,None
3,CrV,4585,1855,1679,None
4,BZ-08,4590,1820,1488,None
5,jeta,4501,1704,1469,None
9,RWi5,4601,1818,1489,None


,型号,长mm,宽mm,高mm,推荐
0,BAOJ310-,4032,1680,1450,0
1,XW-M1+,4330,1535,1885,1
6,POL-,4053,1740,1449,0
7,BW5+,5087,1868,1500,1
8,BDe6+,4560,1822,1645,1
10,Wuling-,3797,1510,1820,0


In [34]:
# (3) 训练数据
train_df = known_df.iloc[:4, :]
train_df_pre = []


test_df = known_df.iloc[4:, :]

display(train_df)
display(test_df)

,型号,长mm,宽mm,高mm,推荐
0,BAOJ310-,4032,1680,1450,0
1,XW-M1+,4330,1535,1885,1
6,POL-,4053,1740,1449,0
7,BW5+,5087,1868,1500,1


,型号,长mm,宽mm,高mm,推荐
8,BDe6+,4560,1822,1645,1
10,Wuling-,3797,1510,1820,0


## 2. 模型训练

（1）创建模型

（2）模型训练

In [35]:
train_df.iloc[:, -1].values

array([0, 1, 0, 1])

In [ ]:
# (1) 创建模型
knn = neighbors.KNeighborsClassifier(n_neighbors=3, weights='distance')

# (2) 模型训练
knn.fit(train_df.iloc[:, 1:-1].values, train_df.iloc[:, -1].values)

# (3) 模型预测
test_df_pre = knn.predict(test_df.iloc[:, 1:-1].values)

# (4) 模型评估
for i in range(len(test_df_pre)):
    print("型号：%s，实际推荐：%d，预测推荐：%d" % (test_df.iloc[i, 0], test_df.iloc[i, -1], test_df_pre[i]))

scores = knn.score(test_df.iloc[:, 1:-1].values, test_df.iloc[:, -1].values)
print("模型准确率：%.2f%%" % (scores * 100))

型号：BDe6+，实际推荐：1，预测推荐：1
型号：Wuling-，实际推荐：0，预测推荐：0
模型准确率：100.00%


## 3. 模型保存与使用

In [ ]:
# # (1) 保存模型
# import joblib
# joblib.dump(knn, "knn_model.pkl")

# # (2) 加载模型
# knn_model = joblib.load("knn_model.pkl")

In [41]:
# 模型使用
results = knn.predict(unknown_df.iloc[:, 1:-1].values)
print("未知型号的推荐结果：")
for i in range(len(results)):
    print("型号：%s，预测推荐：%d" % (unknown_df.iloc[i, 0], results[i]))

未知型号的推荐结果：
型号：GoF，预测推荐：0
型号：CrV，预测推荐：1
型号：BZ-08，预测推荐：1
型号：jeta，预测推荐：0
型号：RWi5，预测推荐：1
